# Make Data Tensors For Drugs and Pseudobulk Data

April 15th 2026

- Load the gene expression data with 20k genes and Morgan fingerprint data and make tensors. 
- Drop drugs with no morgan fingerprints
- Check data for 0 expression rows and drop if any
- Combine repeated data into a single sample (a couple drugs have concentration replicated)
- Combine DMSO samples for all cell lines into 1. (24 ->1)


# Import Processed Data

In [1]:
from pathlib import Path

processed_data_path = Path("data/Tahoe100M_Pseudobulk_processed")

if not processed_data_path.is_absolute():
    for base_path in [Path.cwd(), *Path.cwd().parents]:
        candidate_path = base_path / processed_data_path
        if candidate_path.exists():
            processed_data_path = candidate_path
            break
    else:
        raise FileNotFoundError(f"Could not find {processed_data_path} from {Path.cwd()}")

In [2]:
import anndata as ad
import pandas as pd
from IPython.display import display

data_path = processed_data_path
h5ad_files = sorted(data_path.glob("*.h5ad"))

file_summaries = []
for path in h5ad_files:
    backed_adata = ad.read_h5ad(path, backed="r")
    file_summaries.append(
        {
            "file_name": path.name,
            "size_mb": round(path.stat().st_size / (1024 ** 2), 1),
            "n_obs": backed_adata.n_obs,
            "n_vars": backed_adata.n_vars,
        }
    )
    backed_adata.file.close()

files_df = pd.DataFrame(file_summaries)

print(f"Found {len(h5ad_files)} .h5ad files in {data_path}")
display(files_df)


Found 24 .h5ad files in /Users/aniruddh/Library/CloudStorage/OneDrive-UniversityofUtah/Marth Lab/Deep Learning_Online/deep-learning-final-project/data/Tahoe100M_Pseudobulk_processed


,file_name,size_mb,n_obs,n_vars
0,CVCL_0023.h5ad,95.7,1233,20061
1,CVCL_0028.h5ad,82.9,1067,20061
2,CVCL_0069.h5ad,94.7,1221,20061
3,CVCL_0099.h5ad,92.5,1192,20061
4,CVCL_0131.h5ad,95.6,1232,20061
5,CVCL_0152.h5ad,95.6,1232,20061
6,CVCL_0179.h5ad,95.4,1229,20061
7,CVCL_0218.h5ad,94.2,1214,20061
8,CVCL_0292.h5ad,93.1,1199,20061
9,CVCL_0293.h5ad,95.6,1232,20061


In [3]:
# Import Morgan Fingerprints

fingerprints_path = Path("data/Morganfingerprints.csv")

if not fingerprints_path.is_absolute():
    for base_path in [Path.cwd(), *Path.cwd().parents]:
        candidate_path = base_path / fingerprints_path
        if candidate_path.exists():
            fingerprints_path = candidate_path
            break
    else:
        raise FileNotFoundError(f"Could not find {fingerprints_path} from {Path.cwd()}")

morgan_fingerprints = pd.read_csv(fingerprints_path, index_col=0)
morgan_fingerprints.index = morgan_fingerprints.index.astype(str).str.strip()
morgan_fingerprints["has_fingerprint"] = morgan_fingerprints["morgan_fingerprint"].apply(
    lambda bitstring: isinstance(bitstring, str) and len(bitstring) == 2048 and set(bitstring) <= {"0", "1"}
)
morgan_fingerprint_metadata_df = morgan_fingerprints.rename_axis("drug").reset_index()
missing_fingerprint_drugs_df = (
    morgan_fingerprint_metadata_df.loc[
        ~morgan_fingerprint_metadata_df["has_fingerprint"],
        ["drug", "pubchem_cid", "has_fingerprint"],
    ]
    .reset_index(drop=True)
)
missing_fingerprint_drugs = set(missing_fingerprint_drugs_df["drug"])

print(
    f"Loaded {len(morgan_fingerprints)} Morgan fingerprint rows; {len(missing_fingerprint_drugs)} drugs will be dropped from Morgan and treatment tensors."
)
display(missing_fingerprint_drugs_df)


Loaded 379 Morgan fingerprint rows; 2 drugs will be dropped from Morgan and treatment tensors.


,drug,pubchem_cid,has_fingerprint
0,Sacubitril/Valsartan,NaN,False
1,Verteporfin,NaN,False


## Build Model-Ready Tensor Artifacts


### Shared Helpers and Paths


In [4]:
import ast

import numpy as np
import torch

tensor_artifacts_dir = data_path.parent / "Tahoe100M_tensor_artifacts"
save_tensor_artifacts = False


def parse_condition_string(raw_condition):
    parsed_condition = ast.literal_eval(raw_condition)
    if len(parsed_condition) != 1:
        raise ValueError(f"Expected one drug/concentration entry, found {parsed_condition!r}")

    parsed_drug, concentration, concentration_unit = parsed_condition[0]
    return str(parsed_drug).strip(), float(concentration), str(concentration_unit).strip()


def normalize_condition_key(cell_line, drug, concentration, concentration_unit):
    return "|||".join(
        [
            str(cell_line),
            str(drug),
            format(float(concentration), ".15g"),
            str(concentration_unit),
        ]
    )


def weighted_average_expression(expression_matrix, weights):
    weights = np.asarray(weights, dtype=np.float64)
    if weights.ndim != 1:
        raise ValueError("Weights must be one-dimensional.")

    total_weight = float(weights.sum())
    if total_weight <= 0:
        raise ValueError("Weights must sum to a positive value.")

    weighted_average = expression_matrix.T.dot(weights) / total_weight
    return np.asarray(weighted_average, dtype=np.float32).ravel()


def is_valid_fingerprint(bitstring, n_bits=2048):
    return isinstance(bitstring, str) and len(bitstring) == n_bits and set(bitstring) <= {"0", "1"}


def fingerprint_to_vector(bitstring, n_bits=2048):
    if not is_valid_fingerprint(bitstring, n_bits=n_bits):
        raise ValueError("Morgan fingerprint bitstring is missing or invalid.")

    return np.fromiter((float(bit) for bit in bitstring), dtype=np.float32, count=n_bits)


def is_all_zero_expression(vector):
    return bool(np.isclose(np.asarray(vector), 0.0).all())


print(
    f"save_tensor_artifacts={save_tensor_artifacts}; tensor artifact target directory: {tensor_artifacts_dir}"
)


save_tensor_artifacts=False; tensor artifact target directory: /Users/aniruddh/Library/CloudStorage/OneDrive-UniversityofUtah/Marth Lab/Deep Learning_Online/deep-learning-final-project/data/Tahoe100M_tensor_artifacts


### Validate the Shared 20k-Gene Space


In [5]:
reference_gene_ids = None
reference_file_name = None
gene_order_validation_rows = []

for path in h5ad_files:
    backed_adata = ad.read_h5ad(path, backed="r")
    current_gene_ids = backed_adata.var_names.to_list()

    if reference_gene_ids is None:
        reference_gene_ids = current_gene_ids
        reference_file_name = path.name
        matches_reference = True
    else:
        matches_reference = current_gene_ids == reference_gene_ids

    gene_order_validation_rows.append(
        {
            "file_name": path.name,
            "cell_line": path.stem,
            "n_genes": len(current_gene_ids),
            "matches_reference": matches_reference,
        }
    )
    backed_adata.file.close()

if not all(row["matches_reference"] for row in gene_order_validation_rows):
    mismatched_files = [row["file_name"] for row in gene_order_validation_rows if not row["matches_reference"]]
    raise ValueError(f"Gene order mismatch detected in: {mismatched_files}")

gene_order_validation_df = pd.DataFrame(gene_order_validation_rows)
gene_ids = [str(gene_id) for gene_id in reference_gene_ids]

print(
    f"Validated {len(h5ad_files)} files against {reference_file_name}; each file shares the same {len(gene_ids)} genes in the same order."
)
display(gene_order_validation_df)


Validated 24 files against CVCL_0023.h5ad; each file shares the same 20061 genes in the same order.


,file_name,cell_line,n_genes,matches_reference
0,CVCL_0023.h5ad,CVCL_0023,20061,True
1,CVCL_0028.h5ad,CVCL_0028,20061,True
2,CVCL_0069.h5ad,CVCL_0069,20061,True
3,CVCL_0099.h5ad,CVCL_0099,20061,True
4,CVCL_0131.h5ad,CVCL_0131,20061,True
5,CVCL_0152.h5ad,CVCL_0152,20061,True
6,CVCL_0179.h5ad,CVCL_0179,20061,True
7,CVCL_0218.h5ad,CVCL_0218,20061,True
8,CVCL_0292.h5ad,CVCL_0292,20061,True
9,CVCL_0293.h5ad,CVCL_0293,20061,True


### Summarize Repeated Drug + Concentration Conditions


In [6]:
repeat_summary_frames = []

for path in h5ad_files:
    backed_adata = ad.read_h5ad(path, backed="r")
    obs_df = backed_adata.obs.copy()
    backed_adata.file.close()

    parsed_conditions = [parse_condition_string(raw_condition) for raw_condition in obs_df["drugname_drugconc"].tolist()]
    parsed_condition_df = pd.DataFrame(
        parsed_conditions,
        columns=["parsed_drug", "concentration", "concentration_unit"],
        index=obs_df.index,
    )

    obs_df["drug"] = obs_df["drug"].astype(str).str.strip()

    if not parsed_condition_df["parsed_drug"].eq(obs_df["drug"]).all():
        raise ValueError(f"Parsed drug names did not match `obs['drug']` for {path.name}.")

    obs_df = pd.concat([obs_df, parsed_condition_df], axis=1)

    repeated_conditions = (
        obs_df.groupby(["drug", "concentration", "concentration_unit"], sort=True, dropna=False)
        .agg(repeat_count=("drug", "size"), total_n_cells_used=("n_cells_used", "sum"))
        .reset_index()
    )
    repeated_conditions = repeated_conditions.loc[repeated_conditions["repeat_count"] > 1].copy()
    repeated_conditions.insert(0, "file_name", path.name)
    repeated_conditions.insert(0, "cell_line", path.stem)
    repeat_summary_frames.append(repeated_conditions)

if repeat_summary_frames:
    repeated_condition_counts_df = (
        pd.concat(repeat_summary_frames, ignore_index=True)
        .sort_values(["cell_line", "drug", "concentration"], ignore_index=True)
    )
else:
    repeated_condition_counts_df = pd.DataFrame(
        columns=[
            "cell_line",
            "file_name",
            "drug",
            "concentration",
            "concentration_unit",
            "repeat_count",
            "total_n_cells_used",
        ]
    )

print(
    f"Found {len(repeated_condition_counts_df)} repeated cell-line drug/concentration groups across {repeated_condition_counts_df['cell_line'].nunique()} cell lines."
)
display(repeated_condition_counts_df.head(20))


Found 1439 repeated cell-line drug/concentration groups across 24 cell lines.


,cell_line,file_name,drug,concentration,concentration_unit,repeat_count,total_n_cells_used
0,CVCL_0023,CVCL_0023.h5ad,Adagrasib,0.05,uM,15,33438
1,CVCL_0023,CVCL_0023.h5ad,Afatinib,0.05,uM,3,4515
2,CVCL_0023,CVCL_0023.h5ad,Afatinib,0.50,uM,3,8132
3,CVCL_0023,CVCL_0023.h5ad,Afatinib,5.00,uM,3,5492
4,CVCL_0023,CVCL_0023.h5ad,Almonertinib (mesylate),0.05,uM,2,3273
5,CVCL_0023,CVCL_0023.h5ad,Almonertinib (mesylate),0.50,uM,2,7358
6,CVCL_0023,CVCL_0023.h5ad,Almonertinib (mesylate),5.00,uM,2,3997
7,CVCL_0023,CVCL_0023.h5ad,Belumosudil,0.05,uM,2,4255
8,CVCL_0023,CVCL_0023.h5ad,Belumosudil,0.50,uM,2,6089
9,CVCL_0023,CVCL_0023.h5ad,Belumosudil,5.00,uM,2,2333


### Weighted-Combine DMSO Baselines and Treatment Profiles


In [7]:
dmso_baseline_rows = []
dmso_baseline_vectors = []
treatment_expression_rows = []
treatment_expression_vectors = []
dropped_missing_fingerprint_source_rows = []
zero_expression_source_profile_rows = []
zero_expression_treatment_profile_rows = []
zero_expression_dmso_profile_rows = []

for path in h5ad_files:
    cell_line = path.stem
    adata = ad.read_h5ad(path)

    current_gene_ids = [str(gene_id) for gene_id in adata.var_names.to_list()]
    if current_gene_ids != gene_ids:
        raise ValueError(f"Gene order changed while loading {path.name}.")

    obs_df = adata.obs.copy()
    obs_df["row_index"] = np.arange(adata.n_obs)

    parsed_conditions = [parse_condition_string(raw_condition) for raw_condition in obs_df["drugname_drugconc"].tolist()]
    parsed_condition_df = pd.DataFrame(
        parsed_conditions,
        columns=["parsed_drug", "concentration", "concentration_unit"],
        index=obs_df.index,
    )

    obs_df["drug"] = obs_df["drug"].astype(str).str.strip()

    if not parsed_condition_df["parsed_drug"].eq(obs_df["drug"]).all():
        raise ValueError(f"Parsed drug names did not match `obs['drug']` for {path.name}.")

    obs_df = pd.concat([obs_df, parsed_condition_df], axis=1)
    obs_df["cell_line"] = cell_line

    source_row_sums = np.asarray(adata.X.sum(axis=1)).ravel()
    zero_source_profile_df = obs_df.loc[np.isclose(source_row_sums, 0.0)].copy()
    if not zero_source_profile_df.empty:
        zero_source_profile_df["file_name"] = path.name
        zero_expression_source_profile_rows.extend(
            zero_source_profile_df[
                [
                    "cell_line",
                    "file_name",
                    "drug",
                    "concentration",
                    "concentration_unit",
                    "drugname_drugconc",
                    "n_cells_used",
                ]
            ]
            .rename(columns={"drugname_drugconc": "raw_condition_string"})
            .to_dict("records")
        )

    dmso_rows = obs_df.loc[obs_df["drug"].eq("DMSO_TF")].copy()
    if dmso_rows.empty:
        raise ValueError(f"No DMSO_TF rows found in {path.name}.")

    dmso_vector = weighted_average_expression(
        adata.X[dmso_rows["row_index"].to_numpy()],
        dmso_rows["n_cells_used"].to_numpy(dtype=np.float64),
    )
    if is_all_zero_expression(dmso_vector):
        zero_expression_dmso_profile_rows.append(
            {
                "cell_line": cell_line,
                "file_name": path.name,
                "source_count": int(dmso_rows.shape[0]),
                "total_n_cells_used": int(dmso_rows["n_cells_used"].sum()),
            }
        )

    dmso_baseline_vectors.append(dmso_vector)
    dmso_baseline_rows.append(
        {
            "cell_line": cell_line,
            "file_name": path.name,
            "raw_condition_strings": " || ".join(sorted(dmso_rows["drugname_drugconc"].unique())),
            "source_count": int(dmso_rows.shape[0]),
            "total_n_cells_used": int(dmso_rows["n_cells_used"].sum()),
        }
    )

    treatment_rows = obs_df.loc[~obs_df["drug"].eq("DMSO_TF")].copy()
    dropped_treatment_rows = treatment_rows.loc[treatment_rows["drug"].isin(missing_fingerprint_drugs)].copy()
    if not dropped_treatment_rows.empty:
        dropped_treatment_rows["file_name"] = path.name
        dropped_missing_fingerprint_source_rows.extend(
            dropped_treatment_rows[
                [
                    "cell_line",
                    "file_name",
                    "drug",
                    "concentration",
                    "concentration_unit",
                    "drugname_drugconc",
                    "n_cells_used",
                ]
            ]
            .rename(columns={"drugname_drugconc": "raw_condition_string"})
            .to_dict("records")
        )

    treatment_rows = treatment_rows.loc[~treatment_rows["drug"].isin(missing_fingerprint_drugs)].copy()
    grouped_treatments = treatment_rows.groupby(["drug", "concentration", "concentration_unit"], sort=True, dropna=False)

    for (drug_name, concentration, concentration_unit), group_df in grouped_treatments:
        treatment_vector = weighted_average_expression(
            adata.X[group_df["row_index"].to_numpy()],
            group_df["n_cells_used"].to_numpy(dtype=np.float64),
        )
        if is_all_zero_expression(treatment_vector):
            zero_expression_treatment_profile_rows.append(
                {
                    "cell_line": cell_line,
                    "file_name": path.name,
                    "drug": drug_name,
                    "concentration": float(concentration),
                    "concentration_unit": concentration_unit,
                    "source_count": int(group_df.shape[0]),
                    "total_n_cells_used": int(group_df["n_cells_used"].sum()),
                }
            )

        condition_key = normalize_condition_key(cell_line, drug_name, concentration, concentration_unit)
        treatment_expression_vectors.append(treatment_vector)
        treatment_expression_rows.append(
            {
                "condition_key": condition_key,
                "cell_line": cell_line,
                "file_name": path.name,
                "drug": drug_name,
                "concentration": float(concentration),
                "concentration_unit": concentration_unit,
                "raw_condition_strings": " || ".join(sorted(group_df["drugname_drugconc"].unique())),
                "source_count": int(group_df.shape[0]),
                "total_n_cells_used": int(group_df["n_cells_used"].sum()),
            }
        )

    print(
        f"Aggregated {cell_line}: {dmso_rows.shape[0]} DMSO rows into 1 baseline, dropped {dropped_treatment_rows.shape[0]} treatment rows without Morgan fingerprints, and retained {len(grouped_treatments)} treatment profiles."
    )

dropped_missing_fingerprint_source_rows_df = pd.DataFrame(dropped_missing_fingerprint_source_rows)
if dropped_missing_fingerprint_source_rows_df.empty:
    dropped_missing_fingerprint_source_rows_df = pd.DataFrame(
        columns=[
            "cell_line",
            "file_name",
            "drug",
            "concentration",
            "concentration_unit",
            "raw_condition_string",
            "n_cells_used",
        ]
    )
    dropped_missing_fingerprint_profiles_df = pd.DataFrame(
        columns=[
            "cell_line",
            "file_name",
            "drug",
            "concentration",
            "concentration_unit",
            "source_count",
            "total_n_cells_used",
        ]
    )
    drop_missing_fingerprint_drug_summary_df = pd.DataFrame(
        columns=["drug", "n_expression_profiles_dropped", "n_cell_lines_affected", "total_n_cells_used"]
    )
else:
    dropped_missing_fingerprint_source_rows_df = dropped_missing_fingerprint_source_rows_df.sort_values(
        ["cell_line", "drug", "concentration"],
        ignore_index=True,
    )
    dropped_missing_fingerprint_profiles_df = (
        dropped_missing_fingerprint_source_rows_df.groupby(
            ["cell_line", "file_name", "drug", "concentration", "concentration_unit"],
            dropna=False,
        )
        .agg(source_count=("drug", "size"), total_n_cells_used=("n_cells_used", "sum"))
        .reset_index()
        .sort_values(["cell_line", "drug", "concentration"], ignore_index=True)
    )
    drop_missing_fingerprint_drug_summary_df = (
        dropped_missing_fingerprint_profiles_df.groupby("drug", dropna=False)
        .agg(
            n_expression_profiles_dropped=("drug", "size"),
            n_cell_lines_affected=("cell_line", "nunique"),
            total_n_cells_used=("total_n_cells_used", "sum"),
        )
        .reset_index()
        .sort_values("drug", ignore_index=True)
    )

n_missing_fingerprint_drugs_dropped = len(missing_fingerprint_drugs)
n_expression_profiles_dropped = len(dropped_missing_fingerprint_profiles_df)
n_source_rows_dropped = len(dropped_missing_fingerprint_source_rows_df)
n_cell_lines_with_dropped_profiles = (
    int(dropped_missing_fingerprint_profiles_df["cell_line"].nunique()) if n_expression_profiles_dropped else 0
)

drop_missing_fingerprint_summary_df = pd.DataFrame(
    [
        {"metric": "missing_fingerprint_drugs_dropped", "value": n_missing_fingerprint_drugs_dropped},
        {"metric": "treatment_source_rows_dropped", "value": n_source_rows_dropped},
        {"metric": "treatment_profiles_dropped", "value": n_expression_profiles_dropped},
        {"metric": "cell_lines_affected", "value": n_cell_lines_with_dropped_profiles},
    ]
)

zero_expression_source_profiles_df = pd.DataFrame(zero_expression_source_profile_rows)
if zero_expression_source_profiles_df.empty:
    zero_expression_source_profiles_df = pd.DataFrame(
        columns=[
            "cell_line",
            "file_name",
            "drug",
            "concentration",
            "concentration_unit",
            "raw_condition_string",
            "n_cells_used",
        ]
    )
else:
    zero_expression_source_profiles_df = zero_expression_source_profiles_df.sort_values(
        ["cell_line", "drug", "concentration"],
        ignore_index=True,
    )

zero_expression_treatment_profiles_df = pd.DataFrame(zero_expression_treatment_profile_rows)
if zero_expression_treatment_profiles_df.empty:
    zero_expression_treatment_profiles_df = pd.DataFrame(
        columns=[
            "cell_line",
            "file_name",
            "drug",
            "concentration",
            "concentration_unit",
            "source_count",
            "total_n_cells_used",
        ]
    )
else:
    zero_expression_treatment_profiles_df = zero_expression_treatment_profiles_df.sort_values(
        ["cell_line", "drug", "concentration"],
        ignore_index=True,
    )

zero_expression_dmso_profiles_df = pd.DataFrame(zero_expression_dmso_profile_rows)
if zero_expression_dmso_profiles_df.empty:
    zero_expression_dmso_profiles_df = pd.DataFrame(
        columns=["cell_line", "file_name", "source_count", "total_n_cells_used"]
    )
else:
    zero_expression_dmso_profiles_df = zero_expression_dmso_profiles_df.sort_values(
        ["cell_line"],
        ignore_index=True,
    )

zero_expression_summary_df = pd.DataFrame(
    [
        {"profile_level": "source_rows", "n_zero_profiles": len(zero_expression_source_profiles_df)},
        {
            "profile_level": "aggregated_treatment_profiles",
            "n_zero_profiles": len(zero_expression_treatment_profiles_df),
        },
        {"profile_level": "dmso_baselines", "n_zero_profiles": len(zero_expression_dmso_profiles_df)},
    ]
)

dmso_baseline_df = pd.DataFrame(dmso_baseline_rows)
dmso_order = dmso_baseline_df.sort_values("cell_line").index.to_numpy()
dmso_baseline_df = dmso_baseline_df.loc[dmso_order].reset_index(drop=True)
dmso_baseline_expression_tensor = torch.from_numpy(np.vstack([dmso_baseline_vectors[idx] for idx in dmso_order]).astype(np.float32))
dmso_cell_line_to_index = {cell_line: idx for idx, cell_line in enumerate(dmso_baseline_df["cell_line"])}

if len(dmso_cell_line_to_index) != len(dmso_baseline_df):
    raise ValueError("DMSO baseline cell-line keys are not unique.")

treatment_expression_df = pd.DataFrame(treatment_expression_rows)
treatment_order = treatment_expression_df.sort_values(["cell_line", "drug", "concentration", "concentration_unit"]).index.to_numpy()
treatment_expression_df = treatment_expression_df.loc[treatment_order].reset_index(drop=True)
treatment_expression_tensor = torch.from_numpy(
    np.vstack([treatment_expression_vectors[idx] for idx in treatment_order]).astype(np.float32)
)

if treatment_expression_df["condition_key"].duplicated().any():
    duplicated_keys = treatment_expression_df.loc[treatment_expression_df["condition_key"].duplicated(), "condition_key"].tolist()
    raise ValueError(f"Duplicate treatment condition keys remain after aggregation: {duplicated_keys[:5]}")

treatment_condition_key_to_index = {
    condition_key: idx for idx, condition_key in enumerate(treatment_expression_df["condition_key"])
}

expression_artifact_summary_df = pd.DataFrame(
    {
        "artifact": ["dmso_baselines", "treatment_expressions"],
        "n_rows": [dmso_baseline_expression_tensor.shape[0], treatment_expression_tensor.shape[0]],
        "n_genes": [dmso_baseline_expression_tensor.shape[1], treatment_expression_tensor.shape[1]],
    }
)

print(
    f"Dropped {n_missing_fingerprint_drugs_dropped} drugs without Morgan fingerprints, removing {n_expression_profiles_dropped} treatment profiles across {n_cell_lines_with_dropped_profiles} cell lines."
)
print(
    f"Zero-expression diagnostics: {len(zero_expression_source_profiles_df)} raw rows, {len(zero_expression_treatment_profiles_df)} aggregated treatment profiles, and {len(zero_expression_dmso_profiles_df)} DMSO baselines."
)
print(
    f"Built {dmso_baseline_expression_tensor.shape[0]} DMSO baselines and {treatment_expression_tensor.shape[0]} aggregated treatment profiles after filtering missing-fingerprint drugs."
)
display(drop_missing_fingerprint_summary_df)
display(drop_missing_fingerprint_drug_summary_df)
display(dropped_missing_fingerprint_profiles_df.head())
display(zero_expression_summary_df)
display(expression_artifact_summary_df)
display(dmso_baseline_df.head())
display(treatment_expression_df.head())


Aggregated CVCL_0023: 26 DMSO rows into 1 baseline, dropped 6 treatment rows without Morgan fingerprints, and retained 1125 treatment profiles.
Aggregated CVCL_0028: 22 DMSO rows into 1 baseline, dropped 5 treatment rows without Morgan fingerprints, and retained 982 treatment profiles.
Aggregated CVCL_0069: 26 DMSO rows into 1 baseline, dropped 6 treatment rows without Morgan fingerprints, and retained 1114 treatment profiles.
Aggregated CVCL_0099: 26 DMSO rows into 1 baseline, dropped 6 treatment rows without Morgan fingerprints, and retained 1088 treatment profiles.
Aggregated CVCL_0131: 26 DMSO rows into 1 baseline, dropped 6 treatment rows without Morgan fingerprints, and retained 1124 treatment profiles.
Aggregated CVCL_0152: 26 DMSO rows into 1 baseline, dropped 6 treatment rows without Morgan fingerprints, and retained 1124 treatment profiles.
Aggregated CVCL_0179: 26 DMSO rows into 1 baseline, dropped 6 treatment rows without Morgan fingerprints, and retained 1121 treatment pro

,metric,value
0,missing_fingerprint_drugs_dropped,2
1,treatment_source_rows_dropped,143
2,treatment_profiles_dropped,143
3,cell_lines_affected,24


,drug,n_expression_profiles_dropped,n_cell_lines_affected,total_n_cells_used
0,Sacubitril/Valsartan,72,24,148105
1,Verteporfin,71,24,94799


,cell_line,file_name,drug,concentration,concentration_unit,source_count,total_n_cells_used
0,CVCL_0023,CVCL_0023.h5ad,Sacubitril/Valsartan,0.05,uM,1,1415
1,CVCL_0023,CVCL_0023.h5ad,Sacubitril/Valsartan,0.50,uM,1,3042
2,CVCL_0023,CVCL_0023.h5ad,Sacubitril/Valsartan,5.00,uM,1,1373
3,CVCL_0023,CVCL_0023.h5ad,Verteporfin,0.05,uM,1,1483
4,CVCL_0023,CVCL_0023.h5ad,Verteporfin,0.50,uM,1,1554


,profile_level,n_zero_profiles
0,source_rows,0
1,aggregated_treatment_profiles,0
2,dmso_baselines,0


,artifact,n_rows,n_genes
0,dmso_baselines,24,20061
1,treatment_expressions,26696,20061


,cell_line,file_name,raw_condition_strings,source_count,total_n_cells_used
0,CVCL_0023,CVCL_0023.h5ad,"[('DMSO_TF', 0.0, 'uM')]",26,57448
1,CVCL_0028,CVCL_0028.h5ad,"[('DMSO_TF', 0.0, 'uM')]",22,12555
2,CVCL_0069,CVCL_0069.h5ad,"[('DMSO_TF', 0.0, 'uM')]",26,34591
3,CVCL_0099,CVCL_0099.h5ad,"[('DMSO_TF', 0.0, 'uM')]",26,28611
4,CVCL_0131,CVCL_0131.h5ad,"[('DMSO_TF', 0.0, 'uM')]",26,53435


,condition_key,cell_line,file_name,drug,concentration,concentration_unit,raw_condition_strings,source_count,total_n_cells_used
0,CVCL_0023|||(R)-Verapamil (hydrochloride)|||0....,CVCL_0023,CVCL_0023.h5ad,(R)-Verapamil (hydrochloride),0.05,uM,"[('(R)-Verapamil (hydrochloride)', 0.05, 'uM')]",1,2668
1,CVCL_0023|||(R)-Verapamil (hydrochloride)|||0....,CVCL_0023,CVCL_0023.h5ad,(R)-Verapamil (hydrochloride),0.50,uM,"[('(R)-Verapamil (hydrochloride)', 0.5, 'uM')]",1,1503
2,CVCL_0023|||(R)-Verapamil (hydrochloride)|||5|...,CVCL_0023,CVCL_0023.h5ad,(R)-Verapamil (hydrochloride),5.00,uM,"[('(R)-Verapamil (hydrochloride)', 5.0, 'uM')]",1,2350
3,CVCL_0023|||(S)-Crizotinib|||0.05|||uM,CVCL_0023,CVCL_0023.h5ad,(S)-Crizotinib,0.05,uM,"[('(S)-Crizotinib', 0.05, 'uM')]",1,2396
4,CVCL_0023|||(S)-Crizotinib|||0.5|||uM,CVCL_0023,CVCL_0023.h5ad,(S)-Crizotinib,0.50,uM,"[('(S)-Crizotinib', 0.5, 'uM')]",1,2203


### Convert Morgan Fingerprints to Tensors


In [8]:
morgan_fingerprint_df = morgan_fingerprint_metadata_df.copy()

if morgan_fingerprint_df["drug"].duplicated().any():
    raise ValueError("Drug names in Morgan fingerprints are not unique.")

morgan_fingerprint_df = morgan_fingerprint_df.loc[morgan_fingerprint_df["has_fingerprint"]].reset_index(drop=True)
fingerprint_vectors = [fingerprint_to_vector(bitstring) for bitstring in morgan_fingerprint_df["morgan_fingerprint"]]

morgan_fingerprint_tensor = torch.from_numpy(np.vstack(fingerprint_vectors).astype(np.float32))
morgan_drug_to_index = {drug_name: idx for idx, drug_name in enumerate(morgan_fingerprint_df["drug"])}

print(
    f"Converted {morgan_fingerprint_tensor.shape[0]} Morgan fingerprints into a tensor with {morgan_fingerprint_tensor.shape[1]} bits each after dropping {len(missing_fingerprint_drugs)} drugs without valid fingerprints."
)
display(missing_fingerprint_drugs_df)


Converted 377 Morgan fingerprints into a tensor with 2048 bits each after dropping 2 drugs without valid fingerprints.


,drug,pubchem_cid,has_fingerprint
0,Sacubitril/Valsartan,NaN,False
1,Verteporfin,NaN,False


### Save Tensor Artifacts


In [9]:
repeated_condition_counts_path = tensor_artifacts_dir / "repeated_condition_counts.csv"
dropped_missing_fingerprint_profiles_path = tensor_artifacts_dir / "dropped_missing_fingerprint_profiles.csv"
zero_expression_source_profiles_path = tensor_artifacts_dir / "zero_expression_source_profiles.csv"
zero_expression_treatment_profiles_path = tensor_artifacts_dir / "zero_expression_treatment_profiles.csv"
zero_expression_dmso_profiles_path = tensor_artifacts_dir / "zero_expression_dmso_profiles.csv"
dmso_baselines_path = tensor_artifacts_dir / "dmso_baselines.pt"
treatment_expressions_path = tensor_artifacts_dir / "treatment_expressions.pt"
morgan_fingerprints_path = tensor_artifacts_dir / "morgan_fingerprints.pt"

dmso_baseline_bundle = {
    "expressions": dmso_baseline_expression_tensor,
    "gene_ids": gene_ids,
    "cell_lines": dmso_baseline_df["cell_line"].tolist(),
    "file_names": dmso_baseline_df["file_name"].tolist(),
    "raw_condition_strings": dmso_baseline_df["raw_condition_strings"].tolist(),
    "cell_line_to_index": dmso_cell_line_to_index,
    "source_counts": torch.tensor(dmso_baseline_df["source_count"].to_numpy(), dtype=torch.int64),
    "total_n_cells_used": torch.tensor(dmso_baseline_df["total_n_cells_used"].to_numpy(), dtype=torch.int64),
}

treatment_expression_bundle = {
    "expressions": treatment_expression_tensor,
    "gene_ids": gene_ids,
    "condition_keys": treatment_expression_df["condition_key"].tolist(),
    "cell_lines": treatment_expression_df["cell_line"].tolist(),
    "file_names": treatment_expression_df["file_name"].tolist(),
    "drug_names": treatment_expression_df["drug"].tolist(),
    "concentrations": torch.tensor(treatment_expression_df["concentration"].to_numpy(), dtype=torch.float32),
    "concentration_units": treatment_expression_df["concentration_unit"].tolist(),
    "raw_condition_strings": treatment_expression_df["raw_condition_strings"].tolist(),
    "source_counts": torch.tensor(treatment_expression_df["source_count"].to_numpy(), dtype=torch.int64),
    "total_n_cells_used": torch.tensor(treatment_expression_df["total_n_cells_used"].to_numpy(), dtype=torch.int64),
    "condition_key_to_index": treatment_condition_key_to_index,
}

morgan_fingerprint_bundle = {
    "fingerprints": morgan_fingerprint_tensor,
    "drug_names": morgan_fingerprint_df["drug"].tolist(),
    "pubchem_cids": morgan_fingerprint_df["pubchem_cid"].tolist(),
    "drug_to_index": morgan_drug_to_index,
    "has_fingerprint": torch.tensor(morgan_fingerprint_df["has_fingerprint"].to_numpy(), dtype=torch.bool),
}


saved_artifacts_df = None
if save_tensor_artifacts:
    tensor_artifacts_dir.mkdir(parents=True, exist_ok=True)
    repeated_condition_counts_df.to_csv(repeated_condition_counts_path, index=False)
    dropped_missing_fingerprint_profiles_df.to_csv(dropped_missing_fingerprint_profiles_path, index=False)
    zero_expression_source_profiles_df.to_csv(zero_expression_source_profiles_path, index=False)
    zero_expression_treatment_profiles_df.to_csv(zero_expression_treatment_profiles_path, index=False)
    zero_expression_dmso_profiles_df.to_csv(zero_expression_dmso_profiles_path, index=False)
    torch.save(dmso_baseline_bundle, dmso_baselines_path)
    torch.save(treatment_expression_bundle, treatment_expressions_path)
    torch.save(morgan_fingerprint_bundle, morgan_fingerprints_path)

    saved_artifacts_df = pd.DataFrame(
        [
            {"artifact": "repeated_condition_counts", "path": repeated_condition_counts_path, "size_mb": round(repeated_condition_counts_path.stat().st_size / (1024 ** 2), 3)},
            {"artifact": "dropped_missing_fingerprint_profiles", "path": dropped_missing_fingerprint_profiles_path, "size_mb": round(dropped_missing_fingerprint_profiles_path.stat().st_size / (1024 ** 2), 3)},
            {"artifact": "zero_expression_source_profiles", "path": zero_expression_source_profiles_path, "size_mb": round(zero_expression_source_profiles_path.stat().st_size / (1024 ** 2), 3)},
            {"artifact": "zero_expression_treatment_profiles", "path": zero_expression_treatment_profiles_path, "size_mb": round(zero_expression_treatment_profiles_path.stat().st_size / (1024 ** 2), 3)},
            {"artifact": "zero_expression_dmso_profiles", "path": zero_expression_dmso_profiles_path, "size_mb": round(zero_expression_dmso_profiles_path.stat().st_size / (1024 ** 2), 3)},
            {"artifact": "dmso_baselines", "path": dmso_baselines_path, "size_mb": round(dmso_baselines_path.stat().st_size / (1024 ** 2), 3)},
            {"artifact": "treatment_expressions", "path": treatment_expressions_path, "size_mb": round(treatment_expressions_path.stat().st_size / (1024 ** 2), 3)},
            {"artifact": "morgan_fingerprints", "path": morgan_fingerprints_path, "size_mb": round(morgan_fingerprints_path.stat().st_size / (1024 ** 2), 3)},
        ]
    )

    print(f"Saved tensor artifacts to {tensor_artifacts_dir}")
    display(saved_artifacts_df)
else:
    print(
        f"save_tensor_artifacts is False. Bundles were built in memory but not written to {tensor_artifacts_dir}."
    )


save_tensor_artifacts is False. Bundles were built in memory but not written to /Users/aniruddh/Library/CloudStorage/OneDrive-UniversityofUtah/Marth Lab/Deep Learning_Online/deep-learning-final-project/data/Tahoe100M_tensor_artifacts.


### Spot Checks for Tensor Lookups


In [10]:
example_morgan_index = morgan_fingerprint_bundle["drug_to_index"]["Bortezomib"]
example_treatment_key = normalize_condition_key("CVCL_0023", "Adagrasib", 0.05, "uM")
example_treatment_index = treatment_expression_bundle["condition_key_to_index"][example_treatment_key]
example_dmso_index = dmso_baseline_bundle["cell_line_to_index"]["CVCL_0023"]

dropped_drugs_in_morgan_bundle = sorted(set(missing_fingerprint_drugs) & set(morgan_fingerprint_bundle["drug_names"]))
dropped_drugs_in_treatment_bundle = sorted(set(missing_fingerprint_drugs) & set(treatment_expression_bundle["drug_names"]))

if dropped_drugs_in_morgan_bundle or dropped_drugs_in_treatment_bundle:
    raise ValueError(
        f"Dropped drugs are still present in saved bundles: Morgan={dropped_drugs_in_morgan_bundle}, treatment={dropped_drugs_in_treatment_bundle}"
    )

spot_check_df = pd.DataFrame(
    [
        {
            "lookup": "Morgan fingerprint",
            "key": "Bortezomib",
            "resolved_index": example_morgan_index,
            "vector_length": int(morgan_fingerprint_bundle["fingerprints"][example_morgan_index].numel()),
        },
        {
            "lookup": "Treatment expression",
            "key": example_treatment_key,
            "resolved_index": example_treatment_index,
            "vector_length": int(treatment_expression_bundle["expressions"][example_treatment_index].numel()),
        },
        {
            "lookup": "DMSO baseline",
            "key": "CVCL_0023",
            "resolved_index": example_dmso_index,
            "vector_length": int(dmso_baseline_bundle["expressions"][example_dmso_index].numel()),
        },
    ]
)

print("Dropped drugs with no Morgan fingerprints:", sorted(missing_fingerprint_drugs))
print(
    "Zero-expression profile counts:",
    zero_expression_summary_df.set_index("profile_level")["n_zero_profiles"].to_dict(),
)
display(spot_check_df)


Dropped drugs with no Morgan fingerprints: ['Sacubitril/Valsartan', 'Verteporfin']
Zero-expression profile counts: {'source_rows': 0, 'aggregated_treatment_profiles': 0, 'dmso_baselines': 0}


,lookup,key,resolved_index,vector_length
0,Morgan fingerprint,Bortezomib,1,2048
1,Treatment expression,CVCL_0023|||Adagrasib|||0.05|||uM,54,20061
2,DMSO baseline,CVCL_0023,0,20061
